# From Red Flags to Detection Rules
## An LLM-driven Pipeline for Real-Time GOOSE Intrusion Detection and Prevention

> **Autores:** Lucas A. Martins¹*, Silvio E. Quincozes¹²  
> ¹ Universidade Federal de Uberlândia (UFU) – Uberlândia, Brasil  
> ² Universidade Federal do Pampa (UNIPAMPA) – Alegrete, Brasil  
> `{lucas.martins, sequincozes}@ufu.br`

---

### Resumo

Sistemas de Detecção de Intrusão (IDS) baseados em especificação são amplamente utilizados em subestações IEC 61850, mas dependem de regras criadas manualmente por especialistas. Este notebook apresenta um **pipeline orientado por LLM** que automatiza a geração de regras de detecção para detecção e prevenção de intrusões GOOSE em tempo real.

A abordagem utiliza amostras de comunicação rotuladas para identificar *red flags*, que são transformadas em regras de detecção de intrusão executáveis. A prova de conceito usa o **dataset ERENO** e demonstra que as regras geradas detectam comportamentos anômalos com baixo overhead operacional.

## Índice

1. [Introdução](#1-introdução)
2. [IDS Baseado em Especificação para GOOSE](#2-ids-baseado-em-especificação-para-goose)
3. [Arquitetura Proposta](#3-arquitetura-proposta)
4. [Instalação e Configuração do Ambiente](#4-instalação-e-configuração-do-ambiente)
5. [Ingestão de Dados (ERENO)](#5-ingestão-de-dados-ereno)
6. [Extração de Red Flags via LLM](#6-extração-de-red-flags-via-llm)
7. [Geração de Regras de Detecção](#7-geração-de-regras-de-detecção)
8. [Simulação do Switch Programável](#8-simulação-do-switch-programável)
9. [Avaliação: Capacidade de Detecção](#9-avaliação-capacidade-de-detecção)
10. [Avaliação: Viabilidade em Tempo Real](#10-avaliação-viabilidade-em-tempo-real)
11. [Considerações Finais](#11-considerações-finais)
12. [Referências](#12-referências)

---
## 1. Introdução

Subestações digitais baseadas no padrão **IEC–61850** enfrentam desafios crescentes de cibersegurança, incluindo ataques de:

- **Denial-of-Service (DoS)**
- **Injeção de mensagens (Message Injection)**
- **Mascaramento (Masquerade attacks)**

IDS baseados em especificação são atraentes nesse contexto por seu **baixo overhead computacional** e **interpretabilidade**. No entanto, dependem de regras escritas manualmente por especialistas — processo custoso, difícil de escalar e pouco adaptável.

**Motivação principal:** Automatizar a geração dessas regras usando LLMs a partir de amostras rotuladas do dataset ERENO–IEC–61850.

---
## 2. IDS Baseado em Especificação para GOOSE

O protocolo **GOOSE (Generic Object Oriented Substation Event)**, definido pelo padrão IEC–61850-8-1, suporta operações de proteção e controle em tempo crítico via modelo publisher/subscriber sobre Ethernet.

### Campos relevantes de um frame GOOSE

| Categoria | Campos |
|-----------|--------|
| Estruturais | `dst_mac`, `TPID`, `ethType`, `gooseAppid`, `timeAllowedToLive` |
| Consistência | `gocbRef`, `datSet`, `goID`, `t`, `stNum`, `sqNum` |
| Dinâmica | frequência de mensagens, bytes/s, pacotes/s |

> ⚠️ O GOOSE não foi projetado com mecanismos nativos de segurança robustos, tornando-o vulnerável a ataques de injeção, replay e negação de serviço.

---
## 3. Arquitetura Proposta

O pipeline é composto por **quatro estágios principais**:

```
┌─────────────────┐    ┌──────────────────┐    ┌───────────────────┐    ┌──────────────────┐
│  Dataset GOOSE  │───▶│ Red Flag Extract.│───▶│  Rule Generation  │───▶│ Switch Simulation│
│  Rotulado       │    │ (LLM-based)      │    │  (Python rules)   │    │  (Real-time)     │
│  (ERENO)        │    │                  │    │                   │    │                  │
└─────────────────┘    └──────────────────┘    └───────────────────┘    └──────────────────┘
```

| Estágio | Responsabilidade |
|---------|------------------|
| **1. Source Ingestion** | Carrega dataset, organiza features, prepara prompts estruturados |
| **2. Red Flag Extraction** | LLM identifica padrões suspeitos e inconsistências comportamentais |
| **3. Rule Generation** | Traduz red flags em regras Python executáveis |
| **4. Simulated Deployment** | Executa regras sobre tráfego GOOSE em tempo real simulado |

---
## 4. Instalação e Configuração do Ambiente

In [30]:
# Instalação das dependências (recomenda-se ambiente virtual)
!pip install -r requirements.txt

In [31]:
# 4.3 Imports principais
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd

from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # carrega variáveis do .env na sessão

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Defina GROQ_API_KEY no arquivo .env.")


---
## 5. Ingestão de Dados (ERENO)

O dataset **ERENO–IEC–61850** fornece amostras rotuladas de tráfego GOOSE em condições normais e sob ataque.

Cada amostra contém features de três tipos:
- **Protocol-level:** campos do protocolo GOOSE
- **Temporal:** diferenças de timestamp entre mensagens
- **Derived:** métricas calculadas como `stDiff`, `sqDiff`, `timestampDiff`

In [ ]:
# 5.1 Caminho do dataset
from pathlib import Path

DATASET_PATH = Path("small_dataset/test.csv")

print("Dataset em:", DATASET_PATH.resolve())

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset não encontrado em {DATASET_PATH.resolve()}")

df = pd.read_csv(DATASET_PATH)

print(f"Dataset carregado: {len(df)} amostras, {df.shape[1]} features\n")

print("Primeiras linhas:")
display(df.head())


Dataset em: /home/camilla/sbrc/SBRC_2026/small_dataset/train.csv
Dataset carregado: 207 amostras, 52 features

Primeiras linhas:


,Time,isbA,isbB,isbC,vsbA,vsbB,vsbC,isbARmsValue,isbBRmsValue,isbCRmsValue,...,sqDiff,gooseLengthDiff,cbStatusDiff,apduSizeDiff,frameLengthDiff,timestampDiff,tDiff,timeFromLastChange,delay,class
0,1091.2385,-0.039044,-0.042865,0.079455,-0.042158,0.097473,-0.055315,0.046849,0.050634,0.050901,...,-80.0,0.0,1.0,0.0,0.0,0.116166,-805.919857,707.598591,0.000180,poisoned_high_rate
1,2613.6100,0.254370,-0.280967,-0.033508,-0.087483,-0.018146,0.090788,0.778014,0.109190,0.377037,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.004000,0.000203,grayhole
2,4164.5510,0.532722,0.082284,-0.332637,0.086148,-0.019887,-0.074053,0.945089,0.267017,0.246929,...,0.0,0.0,0.0,0.0,0.0,0.003353,0.000000,1318.944666,0.000195,inverse_replay
3,1688.0143,0.024797,0.042299,-0.066042,0.017338,-0.032020,0.014680,0.014291,0.025667,0.033056,...,1.0,0.0,0.0,0.0,0.0,0.004000,0.000000,0.008000,0.000028,masquerade_fake_fault
4,58947.7730,0.034577,-0.149848,0.067955,-0.096449,0.034230,0.062022,0.108773,0.106597,0.074481,...,0.0,0.0,0.0,0.0,0.0,6.438091,0.000000,58564.134007,0.000684,poisoned_high_rate


In [42]:
# 5.2 Checagem de esquema e distribuição de classes
print("Colunas do dataset:")
for col in df.columns:
    print(f" - {col}")

print("\nDistribuição de classes:")
print(df["class"].value_counts())



IMPORTANT_COLUMNS = [
    # Campos GOOSE “padrão” (conteúdo do PDU)
    # "t",
    # "GooseTimestamp",
    "SqNum",
    "StNum",
    "cbStatus",
    # "ethDst",
    # "ethSrc",
    "ethType",
    # "gooseTimeAllowedtoLive",
    "gooseAppid",
    "gooseLen",
    "TPID",
    "gocbRef",
    "datSet",
    "goID",
    "test",
    "confRev",
    # "ndsCom",
    "numDatSetEntries",
    "APDUSize",
    # "protocol",

    # Features derivadas ligadas a ataques (ERENO / literatura)
    "stDiff",
    "sqDiff",
    "gooseLengthDiff",
    "cbStatusDiff",
    "apduSizeDiff",
    "frameLengthDiff",
    "timestampDiff",
    "tDiff",
    "timeFromLastChange",
    "delay",

    # Rótulo
    "class",
]

# Garante que só usa colunas que existem de fato no df
available_cols = [c for c in IMPORTANT_COLUMNS if c in df.columns]
missing_cols = [c for c in IMPORTANT_COLUMNS if c not in df.columns]

print("Colunas usadas no prompt:")
for c in available_cols:
    print(" -", c)

if missing_cols:
    print("\n[Aviso] Colunas não encontradas no dataset e ignoradas:")
    for c in missing_cols:
        print(" -", c)

df_reduced = df[available_cols].copy()
print(f"\nDataset reduzido: {len(df_reduced)} amostras, {df_reduced.shape[1]} features\n")


Colunas do dataset:
 - Time
 - isbA
 - isbB
 - isbC
 - vsbA
 - vsbB
 - vsbC
 - isbARmsValue
 - isbBRmsValue
 - isbCRmsValue
 - vsbARmsValue
 - vsbBRmsValue
 - vsbCRmsValue
 - isbATrapAreaSum
 - isbBTrapAreaSum
 - isbCTrapAreaSum
 - vsbATrapAreaSum
 - vsbBTrapAreaSum
 - vsbCTrapAreaSum
 - t
 - GooseTimestamp
 - SqNum
 - StNum
 - cbStatus
 - frameLen
 - ethDst
 - ethSrc
 - ethType
 - gooseTimeAllowedtoLive
 - gooseAppid
 - gooseLen
 - TPID
 - gocbRef
 - datSet
 - goID
 - test
 - confRev
 - ndsCom
 - numDatSetEntries
 - APDUSize
 - protocol
 - stDiff
 - sqDiff
 - gooseLengthDiff
 - cbStatusDiff
 - apduSizeDiff
 - frameLengthDiff
 - timestampDiff
 - tDiff
 - timeFromLastChange
 - delay
 - class

Distribuição de classes:
class
poisoned_high_rate        23
grayhole                  23
inverse_replay            23
masquerade_fake_fault     23
masquerade_fake_normal    23
normal                    23
high_StNum                23
random_replay             23
injection                 23
Name: c

---
## 6. Extração de Red Flags via LLM

Nesta etapa, usamos um LLM para inspecionar amostras normais e de ataque do dataset ERENO e extrair *red flags* — padrões que distinguem tráfego malicioso do benigno.


In [ ]:
# Função para pedir somente código Python

def call_llm_for_rules(
    prompt: str,
    model: str = "groq/compound",
) -> str:
    """
    Envia o prompt ao LLM Groq e retorna APENAS código Python de regras.
    O modelo é instruído a não incluir texto explicativo, apenas código válido.
    """
    client = get_groq_client()

    system_msg = (
        "Você é um especialista em segurança IEC 61850 e desenvolvedor Python. "
        "Dado o contexto e as red flags, você deve retornar SOMENTE código "
        "Python válido que define funções de regras de detecção de intrusão "
        "para tráfego GOOSE. Não inclua explicações, comentários extensos ou "
        "markdown. Apenas código Python puro."
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt},
    ]

    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_completion_tokens=2048,
        top_p=1,
    )

    code_text = completion.choices[0].message.content.strip()

    # Em muitos modelos, o código vem com ```python ... ```.
    # Vamos remover cercas de markdown se existirem.
    if code_text.startswith("```"):
        code_text = code_text.strip("`")
        # remove possíveis prefixos como "python\n"
        if code_text.lower().startswith("python"):
            code_text = code_text[len("python"):].lstrip()

    return code_text


In [37]:
# 2. Função para salvar o código em rules.py

def save_rules_to_file(code_text: str, filename: str = "rules.py") -> Path:
    """
    Salva o código Python retornado pelo LLM em um arquivo rules.py.
    """
    output_path = Path(filename).resolve()
    output_path.write_text(code_text, encoding="utf-8")
    print(f"Arquivo de regras salvo em: {output_path}")
    return output_path


In [43]:
# prompt que inclui amostras + instruções para gerar regras
prompt_rules = prepare_llm_prompt(df_reduced, n_normal=5, n_attack=5) + """

Agora, com base nas red flags identificadas implicitamente nesses dados,
gere funções de regras de detecção de intrusão em Python no seguinte formato:

def rule_nome_da_regra(packet: dict) -> bool:
    '''
    Retorna True se o pacote violar esta regra (indica ataque), False caso contrário.
    '''
    # lógica usando campos como 'StNum', 'SqNum', 'timestampDiff', etc.

Importante:
- Não escreva texto explicativo.
- Não escreva comentários longos.
- Retorne apenas código Python válido, com uma ou mais funções rule_*.
"""

code_rules = call_llm_for_rules(prompt_rules)
rules_path = save_rules_to_file(code_rules)


Arquivo de regras salvo em: /home/camilla/sbrc/SBRC_2026/rules.py


In [39]:
from groq import Groq
import os

print("GROQ_API_KEY =", os.getenv("GROQ_API_KEY"))

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("Client criado com sucesso:", client is not None)


GROQ_API_KEY = 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
Client criado com sucesso: True


## 7. Geração de Regras de Detecção (de maneira otimizada)

In [47]:
import time
import json
from pathlib import Path

import pandas as pd
from groq import RateLimitError


# 1. Selecionar amostras normais e por classe
def sample_normal_and_attacks(
    df: pd.DataFrame,
    n_normal: int = 20,
    n_per_attack: int = 10,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Separa um pequeno conjunto de referência normal e amostras por classe de ataque.
    Assume que df['class'] == 'normal' para tráfego legítimo e outros valores são ataques.
    """
    df_normal = df[df["class"] == "normal"].sample(
        n=min(n_normal, (df["class"] == "normal").sum()),
        random_state=42,
    )

    attack_classes = sorted(c for c in df["class"].unique() if c != "normal")
    attack_samples: dict[str, pd.DataFrame] = {}

    for cls in attack_classes:
        df_cls = df[df["class"] == cls]
        attack_samples[cls] = df_cls.sample(
            n=min(n_per_attack, len(df_cls)),
            random_state=42,
        )

    return df_normal, attack_samples


# 2. Prompt base com “referência normal”
def make_normal_reference_block(df_normal: pd.DataFrame) -> str:
    header = (
        "Você é um especialista em segurança de subestações IEC 61850.\n"
        "Abaixo estão amostras de tráfego GOOSE normal; use-as como referência\n"
        "de comportamento legítimo. A coluna 'class' sempre é 'normal'.\n\n"
        "=== AMOSTRAS NORMAIS (REFERÊNCIA) ===\n"
    )
    return header + df_normal.to_string(index=False)


# 3. Prompt por classe de ataque para gerar regras incrementais
def make_attack_prompt_for_rules(
    df_normal_ref: pd.DataFrame,
    df_attack: pd.DataFrame,
    attack_class: str,
) -> str:
    """
    Cria um prompt compacto para uma classe de ataque específica,
    reutilizando a mesma referência de tráfego normal.
    """
    normal_block = make_normal_reference_block(df_normal_ref)

    attack_block = (
        f"\n\n=== AMOSTRAS DE ATAQUE: {attack_class} ===\n"
        + df_attack.to_string(index=False)
    )

    instructions = f"""
    
Com base na diferença entre as amostras NORMAIS (referência)
e as amostras da classe de ataque '{attack_class}', gere funções
DE REGRAS DE DETECÇÃO em Python.

Requisitos:
- Retorne APENAS código Python válido, sem explicações ou markdown.
- Cada regra deve ter a forma:

def rule_{attack_class}_<nome_curto>(packet: dict) -> bool:
    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"
    # lógica usando campos como 'StNum', 'SqNum', 'timestampDiff', etc.

- Use apenas campos presentes nas tabelas (colunas do dataset).
- Foque nas condições que distinguem '{attack_class}' do tráfego normal.
"""
    return normal_block + attack_block + instructions


# 4. Append em rules.py
def append_rules_to_file(code_text: str, filename: str = "rules.py") -> Path:
    """
    Acrescenta novas regras ao arquivo rules.py (cria se não existir).
    """
    path = Path(filename).resolve()
    mode = "a" if path.exists() else "w"
    with path.open(mode, encoding="utf-8") as f:
        if mode == "a":
            f.write("\n\n")
        f.write(code_text.strip() + "\n")
    print(f"Regras adicionadas em: {path}")
    return path


# 5. Wrapper com backoff dinâmico para RateLimitError
def call_llm_for_rules_with_retry(
    prompt: str,
    model: str = "groq/compound",
    max_retries: int = 5,
) -> str:
    """
    Chama o LLM para gerar código de regras, tratando RateLimitError
    com espera dinâmica baseada na mensagem da API.
    """
    client = get_groq_client()

    system_msg = (
        "Você é um especialista em segurança IEC 61850 e desenvolvedor Python. "
        "Dado o contexto e as red flags, você deve retornar SOMENTE código "
        "Python válido. Não inclua explicações, texto em linguagem natural "
        "ou markdown. Apenas código Python puro."
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt},
    ]

    attempt = 0
    while True:
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,
                max_completion_tokens=2048,
                top_p=1,
            )
            code_text = completion.choices[0].message.content.strip()
            if code_text.startswith("```"):
                code_text = code_text.strip("`")
                if code_text.lower().startswith("python"):
                    code_text = code_text[len("python"):].lstrip()
            return code_text

        except RateLimitError as e:
            attempt += 1
            if attempt > max_retries:
                print("[RateLimit] Número máximo de tentativas excedido.")
                raise

            # valor padrão se não conseguirmos extrair da mensagem
            wait_seconds = min(60.0, 2 ** attempt)

            try:
                msg = str(e)
                marker = "Please try again in "
                if marker in msg and "s." in msg:
                    fragment = msg.split(marker, 1)[1]
                    seconds_str = fragment.split("s", 1)
                    wait_seconds = float(seconds_str)
            except Exception:
                pass  # fica com o backoff exponencial

            print(
                f"[RateLimit] Aguardando {wait_seconds:.2f}s antes de tentar "
                f"novamente (tentativa {attempt}/{max_retries})..."
            )
            time.sleep(wait_seconds)


# 6. Pipeline de geração incremental de regras
df_normal_ref, attack_samples = sample_normal_and_attacks(
    df_reduced,  # seu df já filtrado nas colunas GOOSE
    n_normal=20,
    n_per_attack=10,
)

for attack_class, df_attack in attack_samples.items():
    print(f"\n=== Gerando regras para classe: {attack_class} ===")
    prompt_attack = make_attack_prompt_for_rules(df_normal_ref, df_attack, attack_class)

    code_rules = call_llm_for_rules_with_retry(
        prompt_attack,
        model="groq/compound",
        max_retries=5,
    )

    append_rules_to_file(f"## {attack_class}", filename="rules.py")
    append_rules_to_file(code_rules, filename="rules.py")



=== Gerando regras para classe: grayhole ===
Regras adicionadas em: /home/camilla/sbrc/SBRC_2026/rules.py
Regras adicionadas em: /home/camilla/sbrc/SBRC_2026/rules.py

=== Gerando regras para classe: high_StNum ===
[RateLimit] Aguardando 2.00s antes de tentar novamente (tentativa 1/5)...
[RateLimit] Aguardando 4.00s antes de tentar novamente (tentativa 2/5)...
[RateLimit] Aguardando 8.00s antes de tentar novamente (tentativa 3/5)...
[RateLimit] Aguardando 16.00s antes de tentar novamente (tentativa 4/5)...
Regras adicionadas em: /home/camilla/sbrc/SBRC_2026/rules.py
Regras adicionadas em: /home/camilla/sbrc/SBRC_2026/rules.py

=== Gerando regras para classe: injection ===
[RateLimit] Aguardando 2.00s antes de tentar novamente (tentativa 1/5)...
[RateLimit] Aguardando 4.00s antes de tentar novamente (tentativa 2/5)...
[RateLimit] Aguardando 8.00s antes de tentar novamente (tentativa 3/5)...
[RateLimit] Aguardando 16.00s antes de tentar novamente (tentativa 4/5)...
Regras adicionadas em

---
## EXECUÇÃO DAS REGRAS GERADAS

In [51]:
from pathlib import Path
import pandas as pd
import rules as rules# rules.py gerado pelo LLM


# Mesmas colunas “semânticas” que usamos no prompt
# IMPORTANT_COLUMNS = [
#     "t",
#     "GooseTimestamp",
#     "SqNum",
#     "StNum",
#     "cbStatus",
#     "ethDst",
#     "ethSrc",
#     "ethType",
#     "gooseTimeAllowedtoLive",
#     "gooseAppid",
#     "gooseLen",
#     "TPID",
#     "gocbRef",
#     "datSet",
#     "goID",
#     "test",
#     "confRev",
#     "ndsCom",
#     "numDatSetEntries",
#     "APDUSize",
#     "protocol",
#     "stDiff",
#     "sqDiff",
#     "gooseLengthDiff",
#     "cbStatusDiff",
#     "apduSizeDiff",
#     "frameLengthDiff",
#     "timestampDiff",
#     "tDiff",
#     "timeFromLastChange",
#     "delay",
#     "class",
# ]


def load_goose_csv(csv_path: str | Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV não encontrado em {csv_path.resolve()}")

    df = pd.read_csv(csv_path)

    # Garante que só usamos as colunas disponíveis
    cols = [c for c in IMPORTANT_COLUMNS if c in df.columns]
    missing = [c for c in IMPORTANT_COLUMNS if c not in df.columns]

    print("Colunas usadas na análise:")
    for c in cols:
        print(" -", c)

    if missing:
        print("\n[Aviso] Colunas ausentes no CSV (ignoradas):")
        for c in missing:
            print(" -", c)

    return df[cols].copy()


def row_to_packet(row: pd.Series) -> dict:
    """
    Converte uma linha do DataFrame em um dicionário `packet`
    para ser passado às funções de regra em rules.py.
    """
    return row.to_dict()


def apply_rules_to_csv(csv_path: str | Path) -> pd.DataFrame:
    """
    Lê um CSV GOOSE, aplica as regras de rules.py a cada linha
    e retorna um DataFrame com colunas extras indicando violações.
    """
    df = load_goose_csv(csv_path)

    # Descobre todas as funções rule_* definidas em rules.py
    rule_functions = {
        name: fn
        for name, fn in vars(rules).items()
        if callable(fn) and name.startswith("rule_")
    }

    if not rule_functions:
        raise RuntimeError("Nenhuma função rule_* encontrada em rules.py")

    print("\nRegras encontradas:")
    for name in rule_functions:
        print(" -", name)

    # Para cada regra, criamos uma coluna booleana
    for rule_name, rule_fn in rule_functions.items():
        results = []
        for _, row in df.iterrows():
            packet = row_to_packet(row)
            try:
                violated = bool(rule_fn(packet))
            except Exception as e:
                # Em caso de erro na regra, marque como False e log opcional
                violated = False
            results.append(violated)
        df[rule_name] = results

    # Coluna geral “decision” (BLOCK se qualquer regra acusar)
    df["any_rule_violation"] = df[[r for r in rule_functions]].any(axis=1)
    df["decision"] = df["any_rule_violation"].map({True: "BLOCK", False: "ALLOW"})

    return df


# Exemplo de uso
if __name__ == "__main__":
    analyzed_df = apply_rules_to_csv("small_dataset/test.csv")
    print("\nResumo das decisões:")
    print(analyzed_df["decision"].value_counts())


Colunas usadas na análise:
 - SqNum
 - StNum
 - cbStatus
 - ethType
 - gooseAppid
 - gooseLen
 - TPID
 - gocbRef
 - datSet
 - goID
 - test
 - confRev
 - numDatSetEntries
 - APDUSize
 - stDiff
 - sqDiff
 - gooseLengthDiff
 - cbStatusDiff
 - apduSizeDiff
 - frameLengthDiff
 - timestampDiff
 - tDiff
 - timeFromLastChange
 - delay
 - class

Regras encontradas:
 - rule_poisoned_high_rate
 - rule_grayhole
 - rule_inverse_replay
 - rule_masquerade_fake_fault
 - rule_unexpected_sequence
 - rule_unusual_length

Resumo das decisões:
decision
BLOCK    469
ALLOW    324
Name: count, dtype: int64


In [52]:
from pathlib import Path
import pandas as pd
import rules as rules  # rules.py gerado pelo LLM




def load_goose_csv(csv_path: str | Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV não encontrado em {csv_path.resolve()}")

    df = pd.read_csv(csv_path)

    # Garante que só usamos as colunas disponíveis
    cols = [c for c in IMPORTANT_COLUMNS if c in df.columns]
    missing = [c for c in IMPORTANT_COLUMNS if c not in df.columns]

    print("Colunas usadas na análise:")
    for c in cols:
        print(" -", c)

    if missing:
        print("\n[Aviso] Colunas ausentes no CSV (ignoradas):")
        for c in missing:
            print(" -", c)

    return df[cols].copy()


def row_to_packet(row: pd.Series) -> dict:
    """
    Converte uma linha do DataFrame em um dicionário `packet`
    para ser passado às funções de regra em rules.py.
    """
    return row.to_dict()


def apply_rules_to_csv(csv_path: str | Path) -> pd.DataFrame:
    """
    Lê um CSV GOOSE, aplica as regras de rules.py a cada linha
    e retorna um DataFrame com colunas extras indicando violações.
    """
    df = load_goose_csv(csv_path)

    # Descobre todas as funções rule_* definidas em rules.py
    rule_functions = {
        name: fn
        for name, fn in vars(rules).items()
        if callable(fn) and name.startswith("rule_")
    }

    if not rule_functions:
        raise RuntimeError("Nenhuma função rule_* encontrada em rules.py")

    print("\nRegras encontradas:")
    for name in rule_functions:
        print(" -", name)

    # Para cada regra, criamos uma coluna booleana
    for rule_name, rule_fn in rule_functions.items():
        results = []
        for _, row in df.iterrows():
            packet = row_to_packet(row)
            try:
                violated = bool(rule_fn(packet))
            except Exception:
                # Em caso de erro na regra, marque como False
                violated = False
            results.append(violated)
        df[rule_name] = results

    # Coluna geral “decision” (BLOCK se qualquer regra acusar)
    df["any_rule_violation"] = df[[r for r in rule_functions]].any(axis=1)
    df["decision"] = df["any_rule_violation"].map({True: "BLOCK", False: "ALLOW"})

    return df


def confusion_matrix_per_class(df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera uma matriz de confusão por classe real, considerando:
    - Positivo = ataque (qualquer class != 'normal'), decision = BLOCK
    - Negativo = normal (class == 'normal'), decision = ALLOW

    A matriz resultante tem, para cada classe real:
    TP, FP, TN, FN e métricas derivadas.
    """
    if "class" not in df.columns or "decision" not in df.columns:
        raise ValueError("DataFrame precisa das colunas 'class' e 'decision'.")

    classes = sorted(df["class"].unique())
    records = []

    for cls in classes:
        df_cls = df[df["class"] == cls]

        if cls == "normal":
            # normal é negativo (esperado ALLOW)
            tn = (df_cls["decision"] == "ALLOW").sum()
            fp = (df_cls["decision"] == "BLOCK").sum()
            tp = 0
            fn = 0
        else:
            # ataques são positivos (esperado BLOCK)
            tp = (df_cls["decision"] == "BLOCK").sum()
            fn = (df_cls["decision"] == "ALLOW").sum()
            # FP/TN contam impacto em outras classes; aqui deixamos 0 por classe
            fp = 0
            tn = 0

        total = len(df_cls)
        tpr = tp / total if total > 0 else 0.0  # recall para essa classe
        fpr = fp / total if total > 0 else 0.0

        records.append(
            {
                "class": cls,
                "support": total,
                "TP": tp,
                "FP": fp,
                "TN": tn,
                "FN": fn,
                "TPR_recall": tpr,
                "FPR": fpr,
            }
        )

    return pd.DataFrame(records)


# Exemplo de uso
if __name__ == "__main__":
    analyzed_df = apply_rules_to_csv("small_dataset/test.csv")

    print("\nResumo das decisões globais:")
    print(analyzed_df["decision"].value_counts())

    print("\nMatriz de confusão por classe:")
    cm_df = confusion_matrix_per_class(analyzed_df)
    print(cm_df.to_string(index=False))


Colunas usadas na análise:
 - SqNum
 - StNum
 - cbStatus
 - ethType
 - gooseAppid
 - gooseLen
 - TPID
 - gocbRef
 - datSet
 - goID
 - test
 - confRev
 - numDatSetEntries
 - APDUSize
 - stDiff
 - sqDiff
 - gooseLengthDiff
 - cbStatusDiff
 - apduSizeDiff
 - frameLengthDiff
 - timestampDiff
 - tDiff
 - timeFromLastChange
 - delay
 - class

Regras encontradas:
 - rule_poisoned_high_rate
 - rule_grayhole
 - rule_inverse_replay
 - rule_masquerade_fake_fault
 - rule_unexpected_sequence
 - rule_unusual_length

Resumo das decisões globais:
decision
BLOCK    469
ALLOW    324
Name: count, dtype: int64

Matriz de confusão por classe:
                 class  support  TP  FP  TN  FN  TPR_recall      FPR
              grayhole       88  74   0   0  14    0.840909 0.000000
            high_StNum       88  88   0   0   0    1.000000 0.000000
             injection       88  71   0   0  17    0.806818 0.000000
        inverse_replay       88  56   0   0  32    0.636364 0.000000
 masquerade_fake_fault   

---
## 9. Avaliação: Capacidade de Detecção (@TODO PRECISAMOS ATUALIZAR ISSO DEPOIS - MEXI DAQUI PARA CIMA)

Avalia as regras geradas comparando as decisões do simulador com os rótulos verdadeiros do dataset.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def evaluate_detection(results: pd.DataFrame) -> None:
    """
    Calcula e exibe métricas de detecção: precisão, recall, F1 e matriz de confusão.
    """
    if results["true_label"].isna().all():
        print("Rótulos verdadeiros não disponíveis. Forneça o dataset rotulado.")
        return

    y_true = results["true_label"].astype(int)
    y_pred = (results["decision"] == "BLOCK").astype(int)

    print("=== Relatório de Classificação ===")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Ataque"]))

    print("=== Matriz de Confusão ===")
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Real: Normal", "Real: Ataque"],
        columns=["Pred: Normal", "Pred: Ataque"]
    )
    display(cm_df)


evaluate_detection(results_df)

=== Relatório de Classificação ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00         2
      Ataque       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5

=== Matriz de Confusão ===


,Pred: Normal,Pred: Ataque
Real: Normal,2,0
Real: Ataque,0,3


---
## 10. Avaliação: Viabilidade em Tempo Real (@TODO PRECISAMOS ATUALIZAR ISSO DEPOIS - MEXI DAQUI PARA CIMA)


Mede a latência de detecção por pacote para verificar se o overhead é adequado para subestações digitais.

In [ ]:
import time


def benchmark_latency(simulator: GOOSESwitchSimulator,
                      packets: List[dict],
                      n_runs: int = 1000) -> dict:
    """
    Mede a latência média de processamento por pacote.
    
    Args:
        simulator: instância do GOOSESwitchSimulator
        packets: lista de pacotes de teste
        n_runs: número de repetições para estimativa estável
    
    Returns:
        dict com latência média, mínima e máxima em microsegundos
    """
    latencies = []
    for _ in range(n_runs):
        for pkt in packets:
            t0 = time.perf_counter()
            simulator.process_packet(pkt)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1e6)  # microsegundos

    return {
        "mean_us": np.mean(latencies),
        "min_us": np.min(latencies),
        "max_us": np.max(latencies),
        "p99_us": np.percentile(latencies, 99),
    }


latency_stats = benchmark_latency(simulator, test_packets, n_runs=500)

print("=== Latência de Processamento por Pacote ===")
for k, v in latency_stats.items():
    print(f"  {k}: {v:.2f} µs")

=== Latência de Processamento por Pacote ===
  mean_us: 1.08 µs
  min_us: 0.54 µs
  max_us: 547.29 µs
  p99_us: 2.87 µs


---
## 11. Considerações Finais

Este trabalho apresentou um pipeline orientado por LLM que automatiza a geração de regras de detecção para IDS baseados em especificação em subestações IEC 61850.

### Contribuições principais

- **Pipeline plug-and-play** end-to-end: do dataset rotulado às regras executáveis
- **Redução da dependência de especialistas** para criação manual de regras
- **Baixo overhead operacional**: regras Python leves adequadas a ambientes em tempo real
- **Reprodutibilidade**: parâmetros configuráveis e artefatos rastreáveis

### Limitações e trabalhos futuros

- Validação em hardware real de subestação (não apenas simulado)
- Avaliação com mais classes de ataque do dataset ERENO
- Comparação com abordagens de ML supervisionado
- Adaptação automática a novos padrões de ataque via atualização incremental

> **Repositório:** https://github.com/sequincozes/CounselorNode

---
## 12. Referências

- **Commission, I. E.** (2003). Communication networks and systems in substations - Part 8-1: SCSM. IET.

- **Hong, J. and Liu, C.** (2019). Intelligent electronic devices with collaborative intrusion detection systems. *IEEE Transactions on Smart Grid*, 10(1):271–281.

- **Hong, J., Liu, C., and Govindarasu, M.** (2014a). Detection of Cyber Intrusions Using Network-Based Multicast Messages for Substation Automation. *ISGT*, IEEE.

- **Hong, J., Liu, C., and Govindarasu, M.** (2014b). Integrated anomaly detection for cyber security of the substations. *IEEE Transactions on Smart Grid*, 5(4):1643–1653.

- **Kwon, Y. et al.** (2015). A behavior-based intrusion detection technique for smart grid infrastructure. *IEEE Eindhoven PowerTech*.

- **Malik, H., Alotaibi, M. A., and Almutairi, A.** (2022). Cyberattacks identification in IEC 61850 based substation using proximal SVM. *Journal of Intelligent & Fuzzy Systems*, 42(2):1213–1222.

- **Quincozes, S. E. et al.** (2021). A survey on intrusion detection and prevention systems in digital substations. *Computer Networks*, 184:107679.

- **Quincozes, S. E. et al.** (2022). ERENO: An Extensible Tool for Generating Realistic IEC–61850 Intrusion Detection Datasets. PhD thesis, UFF.

- **Yang, Y. et al.** (2016a). Intrusion detection system for IEC 61850 based smart substations. *IEEE PESGM*.

- **Yang, Y. et al.** (2016b). Multidimensional intrusion detection system for IEC 61850-based SCADA networks. *IEEE Transactions on Power Delivery*, 32(2):1068–1078.